# 03 - Data Validation

This notebook tests validation of extracted invoice data before submission to Fakturoid.


In [1]:
from src.document_processor import DocumentProcessor
from src.ai_extractor import AIExtractor
from src.config import config
import json
from datetime import datetime


In [2]:
# Validation functions
def validate_invoice_data(data: dict) -> tuple[bool, list[str]]:
    """Validate extracted invoice data.
    
    Returns:
        Tuple of (is_valid, list_of_errors)
    """
    errors = []
    
    # Check required fields
    required_fields = config.extraction.required_fields
    for field in required_fields:
        if not data.get(field):
            errors.append(f"Missing required field: {field}")
    
    # Validate date formats
    date_fields = ['issue_date', 'due_date']
    for field in date_fields:
        if data.get(field):
            try:
                datetime.fromisoformat(data[field])
            except (ValueError, TypeError):
                errors.append(f"Invalid date format for {field}: {data[field]}")
    
    # Validate numeric fields
    if data.get('total_amount'):
        try:
            float(data['total_amount'])
        except (ValueError, TypeError):
            errors.append(f"Invalid total_amount: {data['total_amount']}")
    
    return len(errors) == 0, errors

print("Validation functions loaded")


Validation functions loaded


In [3]:
# Initialize and extract data from a test invoice
doc_processor = DocumentProcessor(config.directories.invoices)
ai_extractor = AIExtractor(config)

files = doc_processor.list_invoice_files()
if files:
    test_file = files[0]
    print(f"Testing validation on: {test_file.name}\n")
    
    invoice_data = ai_extractor.extract_invoice_data(test_file)
    print("Extracted data:")
    print(json.dumps(invoice_data, indent=2, ensure_ascii=False))
else:
    print("No invoice files found.")
    invoice_data = None


Testing validation on: Alien Isolation.pdf

Extracted data:
{
  "invoice_number": "786940972572357",
  "issue_date": "2025-10-02",
  "supplier_name": "Sony Interactive Entertainment Network Europe Limited",
  "total_amount": 207.25,
  "due_date": null,
  "variable_symbol": null,
  "supplier_address": "10 Great Marlborough Street, London, W1F 7LP",
  "supplier_ico": null,
  "supplier_dic": null,
  "currency": "Kc",
  "tax_amount": 35.97,
  "line_items": [
    {
      "description": "Alien: Isolation (Game)",
      "total": 207.25
    }
  ],
  "notes": "This is not a VAT/GST invoice. This email message has been delivered from a send-only address.",
  "confidence": null,
  "source_file": "Alien Isolation.pdf"
}


In [4]:
# Validate the extracted data
if invoice_data:
    is_valid, errors = validate_invoice_data(invoice_data)
    
    print("\n" + "="*50)
    print("VALIDATION RESULTS")
    print("="*50)
    
    if is_valid:
        print("✓ Data is valid and ready for submission")
    else:
        print("✗ Data has validation errors:")
        for error in errors:
            print(f"  - {error}")
else:
    print("No data to validate")



VALIDATION RESULTS
✓ Data is valid and ready for submission


In [5]:
# Validate all invoices
print("Validating all invoices...\n")

validation_results = []
for invoice_file in files:
    print(f"Validating: {invoice_file.name}")
    try:
        data = ai_extractor.extract_invoice_data(invoice_file)
        is_valid, errors = validate_invoice_data(data)
        
        validation_results.append({
            "filename": invoice_file.name,
            "valid": is_valid,
            "errors": errors,
            "data": data
        })
        
        if is_valid:
            print(f"  ✓ Valid")
        else:
            print(f"  ✗ {len(errors)} error(s)")
            for error in errors:
                print(f"    - {error}")
    except Exception as e:
        print(f"  ✗ Extraction failed: {e}")
        validation_results.append({
            "filename": invoice_file.name,
            "valid": False,
            "errors": [f"Extraction error: {e}"],
            "data": None
        })

# Summary
valid_count = len([r for r in validation_results if r["valid"]])
print(f"\n{'='*50}")
print(f"Summary: {valid_count}/{len(files)} invoices are valid")
print(f"{'='*50}")


Validating all invoices...

Validating: Alien Isolation.pdf
  ✓ Valid
Validating: FP20250158 - OrderSummary202509013059566415002039.png
  ✓ Valid
Validating: OpenAI-Invoice-3D6B9186-0031.pdf
  ✓ Valid
Validating: google-workspace-5369924648.pdf
  ✓ Valid

Summary: 4/4 invoices are valid
